- https://medium.com/neo4j/enhancing-the-accuracy-of-rag-applications-with-knowledge-graphs-ad5e2ffab663 (ready-made langchain llm transformer + QA pipeline)
- https://medium.com/neo4j/neo4j-x-langchain-deep-dive-into-the-new-vector-index-implementation-3bd79601cd2 (wiki_loader + chunks as nodes)

In [ ]:
import os
from langchain_ollama import ChatOllama
from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer

LLM_MODEL_NAME="jaro-jm/llm:latest"
# LLM_MODEL_NAME="SpeakLeash/bielik-11b-v2.3-instruct:Q8_0"
os.environ["HSA_OVERRIDE_GFX_VERSION"]="11.0.0"

llm_text = ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=25000, num_predict=25000)

text = """Donald Trump od początku traktował konflikt w Ukrainie jako nieswoją wojnę. Nie miał cierpliwości ani zrozumienia dla ukraińskich racji. Błędnie zakładał też, że zdoła oderwać Rosję od sojuszu z Chinami. Tymczasem oba kraje mocno zacieśniły współpracę gospodarczą, a Chiny dostarczają dziś Rosjanom maszyny i technologie, na które monopol miał kiedyś Zachód. Moskwa i Pekin pamiętają czasy, gdy były podzielone. Putin i Xi są zgodni, że rozpad Związku Radzieckiego nie był koniecznością, tylko skutkiem błędów Kremla. Rosję i Chiny dzieli przepaść kulturalna, bo Rosjanie nie chcą być Chińczykami, ale to akurat ułatwia relacje. Współpraca z Europą grozi w oczach Putina zarażeniem rosyjskiego społeczeństwa „demokratycznym wirusem”, a w kontaktach z Chinami takiego ryzyka nie ma. Rosja może jednak wpaść w nieodwracalną zależność od Chin. Decydując się na inwazję na Ukrainę, Putin chciał przede wszystkim zasilić ludność Rosji dziesiątkami milionów Słowian, co było dla niego ważniejsze od zdobyczy terytorialnych. W tle są obawy przed chińską dominacją."""

In [ ]:
llm_transformer = LLMGraphTransformer(llm=llm_text, node_properties=True)

graph_documents = llm_transformer.convert_to_graph_documents([Document(text)])
print([doc for doc in graph_documents])

In [ ]:
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

allowed_nodes = [
    "Osoba",
    "Państwo",
    "Organizacja",
    "Sojusz",
    "Konflikt",
    "Lokalizacja",
    "Pojęcie",
    "Wydarzenie",
    "Ideologia",
    "Technologia",
    "SektorGospodarki",
    "Kultura",
    "WydarzenieHistoryczne",
    "Polityka",

    "Prawo",
    "Instytucja",
    "Program",
    "Doktryna",
    "Zasób",
    "Proces",
    "Strategia",
    "Umowa",
    "Rynek",
    "ZjawiskoSpołeczne"
]
allowed_relationships = [
    "ZAATAKOWAŁ",
    "OKUPUJE",
    "BRONI_SIĘ_PRZED",
    "DOSTARCZA_BROŃ",
    "GROZI_UŻYCIEM_SIŁY",
    "ZAKOŃCZYŁ_KONFLIKT",

    "NAKŁADA_SANKCJE_NA",
    "ZAWIERA_SOJUSZ_Z",
    "ZERWAŁ_RELACJE_Z",
    "NEGOCJUJE_Z",
    "WSPIERA_POLITYCZNIE",
    "POTĘPIA_DZIAŁANIA",
    "UZNAJE_NIEPODLEGŁOŚĆ",

    "JEST_PRZYWÓDCĄ",
    "KONTROLUJE",
    "JEST_CZĘŚCIĄ",
    "REPREZENTUJE",
    "ZASTĘPUJE",

    "POWODUJE",
    "ZAPOBIEGA",
    "PRZYSPIESZA",
    "DESTABILIZUJE",
    "UMOŻLIWIA",
    "BLOKUJE",
    "ZAGRAŻA",
    "WYZNACZA_KIERUNEK",

    "FINANSUJE",
    "UZALEŻNIA_EKONOMICZNIE",
    "EKSPORTUJE_DO",
    "PRODUKUJE",
    "WYKORZYSTUJE_TECHNOLOGIĘ",

    "OBAWIA_SIĘ",
    "PLANUJE",
    "SPRZECIWIA_SIĘ",
    "KRYTYKUJE"
]
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    allowed_nodes=allowed_nodes,
    allowed_relationships=allowed_relationships,
    node_properties=True,
    relationship_properties=True,
    strict_mode=False,
    additional_instructions="""Rules:
- Output language: Polish.
- Extract 20-25 most important nodes (entities, concepts, events) and 10-15 most important relationships between them from provided text (fewer if the text does not support more without loss of quality).
- Each triplet (Node, Relation, Node) must be a complete, standalone factual statement.
- Relationship labels must be specific and informative, not vague or generic. Avoid single-word relationships like "UWAŻAŁ", "JEST" etc. where possible.
- Do not create meta-nodes (e.g., Autor, Tekst, Artykuł). First-person statements should be reduced to the **underlying objective fact** connecting real entities.""",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(text)])

In [ ]:
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")

In [ ]:
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    node_properties=True,
    relationship_properties=True,
    strict_mode=False,
    additional_instructions="""Rules:
- Output language: Polish.
- Extract 20-25 most important nodes (entities, concepts, events) and 10-15 most important relationships between them from provided text (fewer if the text does not support more without loss of quality).
- Each triplet (Node, Relation, Node) must be a complete, standalone factual statement.
- Relationship labels must be specific and informative, not vague or generic. Avoid single-word relationships like "UWAŻAŁ", "JEST" etc. where possible.
- Do not create meta-nodes (e.g., Autor, Tekst, Artykuł). First-person statements should be reduced to the **underlying objective fact** connecting real entities.""",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(text)])

In [ ]:
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")